In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from utils import *

# 通过凹缺陷检测的方法找到液滴曲线的范围

In [ ]:
#读取原3通道图像，转换成灰度格式
img1_3ch=cv2.imread('1.jpg')
img1=cv2.cvtColor(img1_3ch,cv2.COLOR_BGR2GRAY)
print('image size: ',img1.shape)

In [ ]:
plt.imshow(img1,cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
# 高斯模糊，并用Canny找轮廓
bl=cv2.GaussianBlur(img1,(9,9),4)
ca=cv2.Canny(bl,50,60)
threshed=cv2.dilate(ca,(15,15),iterations=5)
plt.imshow(threshed,cmap='gray') 
plt.show()

In [ ]:
# 用findContour读取连续轮廓坐标，并找到最长的那一条
cnts,hier=cv2.findContours(threshed,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
denoised=threshed.copy()
lengths=np.zeros(len(cnts))
for t,c in enumerate(cnts):
    lengths[t]=cv2.arcLength(c,False)
Curve=cnts[np.argmax(lengths)]

In [ ]:
# 展示查找结果
img_contour=img1_3ch.copy()
img_contour=cv2.drawContours(img_contour,[Curve],-1,(0,255,0),4)

b,g,r=cv2.split(img_contour)
img_contour=cv2.merge([r,g,b])

fig=plt.figure(figsize=(8,6))
plt.imshow(img_contour)
plt.axis('off')
plt.savefig('./CurveDetected')
plt.show()

In [ ]:
#原图像分辨率较高，得到的轮廓点过于密集，为节省计算资源，做down-sampling
original_length=len(Curve)
choice=np.linspace(0,original_length,
                   1000 if original_length>1000 else original_length,
                   endpoint=False,dtype=np.int32)


print('Down-sampling 之前的曲线点数：',len(Curve))
Curve=Curve[choice]
print('Down-sampling 之后的曲线点数：',len(Curve))

In [ ]:
#contour的坐标纵轴以下方为正，所以其中纵坐标最大的点就是液滴的底点

#contour的储存顺序为逆时针，所以在数组中，底点之后的一些点为右半轮廓，底点之前的一些点为左半轮廓

#PointsNUM=int(Curve.shape[0]*0.4)
tip_index=np.argmax(Curve[::,0,1])
print('液滴底点index：',tip_index)

#为凹缺陷检测制作凸包
hull_points=cv2.convexHull(Curve,returnPoints=True)
hull_indices=cv2.convexHull(Curve,returnPoints=False)
hull_indices[::-1].sort(axis=0)

In [ ]:
convexDefects=cv2.convexityDefects(Curve,hull_indices)
convexDef_length=convexDefects[::,0,1]-convexDefects[::,0,0]
# (convexDef_indices,)=np.where(convexDef_length>50)
convexDef_indices=np.argsort(convexDef_length)[-3:-1]

convexDef_length

In [ ]:
# 取出左右分界点。这里算法我没完善好，得人工判断。
# 上面cell的输出，较大的接近的两个值即是两个分界点
# 把人工选出的两个位置输入在这
a=2 # 第二个
b=-2 # 倒数第三个

a = min(a, convexDefects.shape[0] - 1)
b = min(b, convexDefects.shape[0] - 1)
left_index,right_index=convexDefects[[a,b],0,2]

print('左分界点的index: ',left_index)
print('右分界点的index: ',right_index)

In [ ]:
# 展示分割结果
img_contour=img1_3ch.copy()
img_contour=cv2.drawContours(img_contour,[Curve],-1,(0,255,0),25)
img_contour=cv2.drawContours(img_contour,[hull_points],-1,(255,40,85),25)
img_contour=cv2.circle(img_contour,Curve[left_index].squeeze(),20,color=(0,80,255),thickness=-1)
img_contour=cv2.circle(img_contour,Curve[right_index].squeeze(),20,color=(0,80,255),thickness=-1)
b,g,r=cv2.split(img_contour)
img_contour=cv2.merge([r,g,b])

fig=plt.figure(figsize=(8,6))
#plt.suptitle('Convex hull and the original curve')
plt.axis('off')
fig.savefig('./ConvexHull_1.jpg')
plt.imshow(img_contour)
plt.show()

In [ ]:
#截取轮廓中属于液滴轮廓的部分
l,r=sorted([left_index,right_index])
PendantCurve=Curve[l:r+1]

#计算左分界点与右分界点之间的距离，即可得到毛细管直径
diameter=np.sum(np.square(PendantCurve[0,0,::]-PendantCurve[-1,0,::]))
diameter=np.sqrt(diameter)
print('毛细管直径为：',diameter,'个像素')

In [ ]:
img_contour=img1_3ch.copy()
img_contour=cv2.drawContours(img_contour,[PendantCurve],-1,(0,255,0),10)
b,g,r=cv2.split(img_contour)
img_contour=cv2.merge([r,g,b])

fig=plt.figure(figsize=(8,6))
#plt.suptitle('The detected silhouette of pendant')
plt.imshow(img_contour)
plt.axis('off')
fig.savefig('./DetectedCurve')
plt.show()

In [ ]:
#以液滴底点作为（0，0）,并做上下翻转
PendantCurve_numerical=(PendantCurve-PendantCurve[tip_index-left_index]).reshape(-1,2)
PendantCurve_numerical=-np.array(PendantCurve_numerical,dtype=np.float32)

print(PendantCurve_numerical.shape)

In [ ]:
#最终得到的轮廓点
from matplotlib import rcParams
fig=plt.figure(figsize=(8,6))
rcParams['font.size']=10
r,z=PendantCurve_numerical.T
r = r[z<=500]
z = z[z<=500]
plt.scatter(r,z,s=2.0)
plt.axis('equal')
plt.xlabel(r'$x$')
plt.ylabel(r'$y$')
fig.savefig('./FinalCurve.jpg')
plt.show()

In [ ]:
curve_data = np.hstack((r.reshape(-1,1), z.reshape(-1,1)))
np.save('curve_data.npy', curve_data)

In [ ]:
# 保存读取到的轮廓数据
import pickle
with open('./Exp1.pkl','wb') as f:
    pickle.dump(PendantCurve_numerical,f)